# **Data Cleaning and Feature Extraction**

## Load Dataset

### Mount Google Drive

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Define Dataset Path

In [27]:
import os

DRIVE_DATASET_PATH = "/content/drive/MyDrive/ML_Project/Dataset"

# Verify dataset exists
if os.path.exists(DRIVE_DATASET_PATH):
    print("Dataset found in Drive.")
    print("Languages available:", os.listdir(DRIVE_DATASET_PATH))
else:
    print("Dataset path not found. Check the path.")


Dataset found in Drive.
Languages available: ['Italian', 'Korean', 'German', 'Spanish']


### Copy Dataset to Local Colab Storage

In [ ]:
LOCAL_DATASET_PATH = "/content/Dataset"

print("Copying dataset to local runtime...")
!cp -r "/content/drive/MyDrive/ML_Project/Dataset" /content/
print("Copy complete.")

Copying dataset to local runtime...
Copy complete.


### Verify Local Copy

In [29]:
if os.path.exists(LOCAL_DATASET_PATH):
    print("Local dataset ready.")
    print("Languages:", os.listdir(LOCAL_DATASET_PATH))
else:
    print("Local dataset not found.")


Local dataset ready.
Languages: ['Italian', 'Korean', 'German', 'Spanish']


## Data Cleaning


In this section we:
- Standardize sampling rate to 16kHz
- Convert audio to mono
- Trim leading and trailing silence
- Normalize amplitude

### Build File List

In [31]:
import os
import numpy as np
import librosa
from tqdm import tqdm

DATASET_PATH = "/content/Dataset"

languages = ["German", "Italian", "Korean", "Spanish"]
genders = ["Male", "Female"]
file_paths = []
labels = []

for lang in languages:
    # Special case for German
    if lang == "German":
        gender_folders = ["female90", "male90"]
    else:
        gender_folders = genders

    for gender in gender_folders:
        folder_path = os.path.join(DATASET_PATH, lang, gender)
        for file in os.listdir(folder_path):
            if file.endswith(".mp3"):
                file_paths.append(os.path.join(folder_path, file))
                labels.append(lang)

print("Total audio files:", len(file_paths))


Total audio files: 712


### Define Cleaning Function

In [32]:
def clean_audio(file_path, target_sr=16000):
    # Load audio (resample + mono)
    signal, sr = librosa.load(file_path, sr=target_sr, mono=True)

    # Trim leading & trailing silence
    signal, _ = librosa.effects.trim(signal, top_db=30)

    # Normalize amplitude
    signal = signal / np.max(np.abs(signal))

    return signal, target_sr


### Data Augmentation Functions

In [33]:
def add_noise(signal, noise_factor=0.005):
    noise = np.random.randn(len(signal))
    augmented = signal + noise_factor * noise
    return augmented

def time_shift(signal, shift_max=0.2):
    shift = int(np.random.uniform(-shift_max, shift_max) * len(signal))
    return np.roll(signal, shift)

def pitch_shift(signal, sr, n_steps=2):
    return librosa.effects.pitch_shift(signal, sr=sr, n_steps=n_steps)


## Feature Extraction

We now convert the waveform into a numerical representation.

We use the following features:

- **MFCC (13 coefficients)**
- **Chroma**
- **Spectral centroid**
- **Spectral bandwidth**
- **Zero-crossing rate**

For each feature, we compute:

- **Mean**
- **Standard deviation**


In [34]:
def extract_features(signal, sr):
    features = []

    # MFCC
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=13)
    features.extend(np.mean(mfcc, axis=1))
    features.extend(np.std(mfcc, axis=1))

    # Chroma
    chroma = librosa.feature.chroma_stft(y=signal, sr=sr)
    features.extend(np.mean(chroma, axis=1))
    features.extend(np.std(chroma, axis=1))

    # Spectral centroid
    spec_centroid = librosa.feature.spectral_centroid(y=signal, sr=sr)
    features.append(np.mean(spec_centroid))
    features.append(np.std(spec_centroid))

    # Spectral bandwidth
    spec_bw = librosa.feature.spectral_bandwidth(y=signal, sr=sr)
    features.append(np.mean(spec_bw))
    features.append(np.std(spec_bw))

    # Zero crossing rate
    zcr = librosa.feature.zero_crossing_rate(signal)
    features.append(np.mean(zcr))
    features.append(np.std(zcr))

    return np.array(features)


## Full Processing Pipeline

We will:
- Clean the data
- Extract original features  
- Optionally create one augmented version  
- Store **features only** (not raw signals)

In [ ]:
X = []
y = []

total_files = len(file_paths)
print(f"Processing {total_files} audio files...\n")

for path, label in tqdm(zip(file_paths, labels),
                        total=total_files,
                        desc="Processing",
                        unit="file"):

    # Clean original
    signal, sr = clean_audio(path)

    # Extract features from original
    features = extract_features(signal, sr)
    X.append(features)
    y.append(label)

    # Optional Augmentation
    # Aug 1: Add noise
    # augmented_signal = add_noise(signal)
    # ...
    # features_aug = extract_features(augmented_signal, sr)
    # X.append(features_aug)
    # y.append(label)

print("\nProcessing complete.")


Processing 712 audio files...



Processing: 100%|██████████| 712/712 [06:10<00:00,  1.92file/s]


Processing complete.


## Save Features

In [36]:
X = np.array(X)
y = np.array(y)
np.save("X.npy", X)
np.save("y.npy", y)

print("Features saved locally.")


Features saved locally.
